# Pairs Trading — Backtesting the Strategy

---

## What is a Backtest, and What Does it Tell You?

A backtest replays a trading strategy against historical data to estimate how it *would have* performed. You feed in a signal series — the entry and exit triggers produced by the z-score logic in Notebook 01 — and simulate every trade as if you had actually executed it in real time.

The output answers questions the signal notebook cannot:
- **Would you have made money?** (total return, CAGR)
- **How smooth was the ride?** (Sharpe ratio, max drawdown)
- **Were most individual trades profitable?** (win rate, avg win/loss)
- **Was the overall edge positive?** (profit factor)

### What a backtest does NOT tell you

A backtest is a historical simulation, not a prediction. Several important caveats:

**Look-ahead bias** — If you use any information from the future to generate signals (e.g., fitting the hedge ratio on all data before using it), your results will be optimistic. This notebook uses a *static* hedge ratio estimated on the full window, which is a known simplification. Walk-forward testing (train/test splits, then roll forward) eliminates this.

**Overfitting** — If you tune parameters (z-entry, rolling window, etc.) until the backtest looks great, you have learned the noise in the historical data, not a real edge. Any strategy should be validated on data it was never trained on.

**Market impact** — Backtests assume you can trade any size at the current close price. In reality, large orders move prices. For the small position sizes here ($10k/leg) this is negligible, but it is worth knowing.

**Regime change** — A strategy that worked from 2020-2024 may not work in 2025-2026 if the underlying relationship between the stocks has changed. The ADF p-value is an important signal to monitor.

This backtest uses **close-to-close fills** — you enter at the close on the signal bar and exit at the close on the exit bar. We will discuss this limitation in Section 6.

---

## Roadmap for this Notebook

1. **Run the signal pipeline** — fetch data, compute spread, generate z-score signals (brief recap of Notebook 01)
2. **Build the trade log** — understand how position sizing works and inspect every trade
3. **Build the equity curve** — simulate P&L over time; understand the cumulative sum approach
4. **Compute performance metrics** — work through the math of each statistic
5. **Visualize the results** — equity curve, trade P&L, metrics table, explanation page
6. **Understand the limitations** — what would need to change for a production backtest

In [ ]:
import sys
sys.path.insert(0, '../..')  # make src/ importable from notebooks/pairs/

from src.data.fetcher import fetch_pair, fetch_pair_ohlcv
from src.strategies.pairs.config import DEFAULT_PARAMS
from src.strategies.pairs.spread import compute_hedge_ratio, compute_spread
from src.analytics.stationarity import adf_test, compute_half_life
from src.strategies.pairs.signals import compute_zscore, generate_signals
from src.strategies.pairs.backtest import build_trade_log, run_pairs_backtest
from src.backtest.engine import run_backtest
from src.backtest.metrics import compute_metrics
from src.strategies.pairs.viz import (
    plot_equity_curve, plot_trade_pnl,
    plot_backtest_metrics, plot_backtest_interpretation,
)

import pandas as pd
import numpy as np
import math

---

## Section 1: Signal Pipeline Recap

This section runs the full signal pipeline from Notebook 01 in compressed form. Refer back to that notebook for detailed explanations of each step.

We will use **KO/PEP** as the working example throughout this notebook. Note that as covered in Notebook 01, neither KO/PEP nor any of the currently configured pairs passes the ADF stationarity test over the 2-year window — KO has risen roughly 32% while PEP has fallen roughly 10%, producing structural drift rather than mean-reversion.

**This means the backtest results will likely be negative.** That is actually the most valuable thing the backtest can show us — it confirms that trading a non-stationary spread is unprofitable, exactly as the theory predicts. The goal here is to learn how to *read* backtest output, not to find a profitable strategy on this particular pair.

Once you have valid pairs (shorter lookback, or pairs that genuinely cointegrate), you can re-run this notebook and interpret the results in the same framework.

In [ ]:
TICKER_A = 'KO'
TICKER_B = 'PEP'
p = DEFAULT_PARAMS

# Full signal pipeline
df      = fetch_pair(TICKER_A, TICKER_B, period=p['period'])
beta    = compute_hedge_ratio(df['close_a'], df['close_b'])
spread  = compute_spread(df['close_a'], df['close_b'], beta)
adf     = adf_test(spread)
hl      = compute_half_life(spread)
zscore  = compute_zscore(spread, window=p['rolling_window'])
signals = generate_signals(zscore, z_entry=p['z_entry'], z_exit=p['z_exit'], z_stop=p['z_stop'])

print(f"Pair           : {TICKER_A}/{TICKER_B}")
print(f"Data range     : {df.index[0].date()} to {df.index[-1].date()}  ({len(df)} trading days)")
print(f"Hedge ratio b  : {beta:.4f}")
print(f"ADF p-value    : {adf['p_value']:.4f}  ({'stationary' if adf['is_stationary'] else 'NOT stationary'})")
print(f"Half-life      : {hl:.1f} trading days")
print()
print("Signal counts:")
for event, count in signals['signal'].value_counts().dropna().items():
    print(f"  {event:<18}: {count}")

---

## Section 2: The Trade Log

The trade log is the bridge between the signal series and the backtest engine. It contains one row per *completed trade* — every entry that was later closed by either an exit or a stop signal.

### What `build_trade_log` does

The function iterates through the signal DataFrame in chronological order, maintaining an `open_trade` state variable:

1. When it sees a `long_spread` or `short_spread` signal and no position is open, it records the entry details.
2. When it sees an `exit` or `stop` signal with a position open, it records the exit details, computes P&L, and appends the completed trade to the list.
3. If an entry signal fires while already in a position, it is ignored — the state machine prevents overlapping trades.

The result is a clean table where each row represents the full lifecycle of a single trade.

### Position sizing: dollar-neutral, hedge-ratio adjusted

Each leg of the trade is sized to approximately $10,000 (default `capital_per_leg`), adjusted by the hedge ratio so the spread exposure is balanced:

**Long spread** (spread is unusually low — buy KO, short PEP):
$$\text{shares}_A = +\frac{\text{capital}}{\text{close}_A}$$
$$\text{shares}_B = -\frac{\text{capital} \times \beta}{\text{close}_B}$$

**Short spread** (spread is unusually high — short KO, buy PEP):
$$\text{shares}_A = -\frac{\text{capital}}{\text{close}_A}$$
$$\text{shares}_B = +\frac{\text{capital} \times \beta}{\text{close}_B}$$

The negative sign on the B leg ensures the two legs offset each other in dollar terms. If $\beta = 1$ and both stocks trade at the same price, you would hold exactly the same dollar value long on one side and short on the other — perfectly dollar-neutral. The hedge ratio $\beta$ adjusts for the different log-price sensitivities.

### P&L calculation

$$\text{P\&L} = \text{shares}_A \times (\text{exit}_A - \text{entry}_A) + \text{shares}_B \times (\text{exit}_B - \text{entry}_B)$$

Positive P&L on the long leg means the stock we owned went up. Positive P&L on the short leg means the stock we shorted went down. Both contribute to profit when the spread reverts as expected.

The `pnl_pct` column expresses this as a percentage of total capital committed:
$$\text{pnl\_pct} = \frac{\text{P\&L}}{2 \times \text{capital\_per\_leg}}$$

Since total deployed capital is 2x the per-leg amount (one long leg + one short leg), dividing by $2 \times \text{capital}$ gives the return on invested capital for that trade.

In [ ]:
df_ohlcv = fetch_pair_ohlcv(TICKER_A, TICKER_B, period=p['period'])

trades = build_trade_log(
    signals_df=signals,
    df_ohlcv=df_ohlcv,
    hedge_ratio=beta,
    capital_per_leg=10_000.0,
)

print(f"Total completed trades: {len(trades)}")
print()

if trades.empty:
    print("No trades to show. Try lowering --z-entry or using a different pair.")
else:
    pd.set_option('display.float_format', '{:.4f}'.format)
    display_cols = ['entry_date', 'exit_date', 'exit_type', 'direction',
                    'entry_price_a', 'exit_price_a', 'entry_price_b', 'exit_price_b',
                    'pnl', 'pnl_pct']
    print(trades[display_cols].to_string(index=False))

**Understanding the trade log columns:**

| Column | Meaning |
|---|---|
| `direction` | +1 = long spread (long A, short B), -1 = short spread (short A, long B) |
| `exit_type` | `exit` = profitable close near z=0; `stop` = loss-limiting close at z=±3 |
| `entry_price_a/b` | Closing price of each stock on the entry bar |
| `exit_price_a/b` | Closing price of each stock on the exit bar |
| `shares_a` | Share count for leg A (positive = long, negative = short) |
| `shares_b` | Share count for leg B (positive = long, negative = short) |
| `pnl` | Gross dollar P&L for the trade |
| `pnl_pct` | P&L as a fraction of total deployed capital (2 x $10k = $20k) |

**What to notice:**

- The `direction` of `shares_a` and `shares_b` should always be *opposite* — this is the dollar-neutral condition.
- `exit` trades should generally have smaller P&L magnitudes (the spread barely moved before reverting). `stop` trades will have larger losses (the spread moved significantly against us).
- If all trades are stops and no exits, the spread is trending rather than mean-reverting — a strong signal that the pair is not stationary.

In [ ]:
# Manual P&L verification of the first trade.
# Confirms the formula: pnl = shares_a * (exit_a - entry_a) + shares_b * (exit_b - entry_b)

if not trades.empty:
    t = trades.iloc[0]
    entry_date = t['entry_date'].date() if hasattr(t['entry_date'], 'date') else t['entry_date']
    exit_date  = t['exit_date'].date()  if hasattr(t['exit_date'],  'date') else t['exit_date']

    print(f"Trade 0: {TICKER_A}/{TICKER_B}  |  {entry_date} -> {exit_date}  |  {t['exit_type']}")
    print()
    print(f"  shares_a = {t['shares_a']:+.4f}  (direction {t['direction']:+d})")
    print(f"  shares_b = {t['shares_b']:+.4f}")
    print()
    leg_a_move = t['exit_price_a'] - t['entry_price_a']
    leg_b_move = t['exit_price_b'] - t['entry_price_b']
    leg_a_pnl  = t['shares_a'] * leg_a_move
    leg_b_pnl  = t['shares_b'] * leg_b_move
    computed   = leg_a_pnl + leg_b_pnl
    print(f"  Leg A: {t['shares_a']:+.4f} shares  x  ({t['exit_price_a']:.4f} - {t['entry_price_a']:.4f}) = {leg_a_pnl:+.4f}")
    print(f"  Leg B: {t['shares_b']:+.4f} shares  x  ({t['exit_price_b']:.4f} - {t['entry_price_b']:.4f}) = {leg_b_pnl:+.4f}")
    print()
    print(f"  Computed P&L : {computed:+.4f}")
    print(f"  Stored P&L   : {t['pnl']:+.4f}")
    print(f"  Match        : {abs(computed - t['pnl']) < 0.01}")
else:
    print("No trades to verify.")

---

## Section 3: The Equity Curve

The equity curve tracks portfolio value over time. It starts at the starting capital and moves up or down as each trade closes with a gain or loss.

### How `run_backtest` constructs the equity curve

The function takes the trade log and applies this algorithm:

**Step 1 — Group P&L by exit date:**
```python
pnl_by_date = trades.groupby("exit_date")["pnl"].sum()
```
Multiple trades can close on the same day (rare in a state machine that prevents overlapping positions). Summing ensures all same-day P&L is captured.

**Step 2 — Create a full business-day date range:**
```python
all_dates = pd.date_range(start=first_entry, end=last_exit, freq="B")
```
The `freq="B"` means business days only (no weekends or holidays). We need a continuous series — not just the days with trade activity — so we can build a daily equity curve.

**Step 3 — Reindex and fill zeros:**
```python
pnl_series = pnl_by_date.reindex(all_dates, fill_value=0.0)
```
Days with no trade activity get P&L of 0. This is accurate: when you are in an open position, your unrealized P&L is not recorded yet — it only crystallizes when you exit.

**Step 4 — Cumulative sum:**
```python
equity = starting_capital + pnl_series.cumsum()
```
`cumsum()` converts daily P&L into a running total. Adding starting capital gives the actual portfolio value on each day.

### What this approach captures (and misses)

**Captured:** The dollar impact of every completed trade; flat periods where no trades are active; the sequence and timing of gains and losses.

**Not captured:** Unrealized P&L on open positions (the equity line is flat during a live trade, then jumps at exit); intraday price movements within a trade; the opportunity cost of capital deployed in open positions.

This is a **realized-P&L equity curve**. The portfolio value only changes when a trade closes. During an open trade, the chart shows a flat line even though the position's unrealized P&L is fluctuating daily. This is a common and acceptable simplification for daily-bar backtests.

**Starting capital:** Each run uses $10,000/leg × 2 legs = $20,000. All return metrics are relative to this baseline.

In [ ]:
CAPITAL_PER_LEG  = 10_000.0
STARTING_CAPITAL = 2 * CAPITAL_PER_LEG  # $20,000

trades, equity_curve, metrics = run_pairs_backtest(
    ticker_a=TICKER_A,
    ticker_b=TICKER_B,
    signals_df=signals,
    df_ohlcv=df_ohlcv,
    hedge_ratio=beta,
    capital_per_leg=CAPITAL_PER_LEG,
)

print(f"Equity curve: {len(equity_curve)} business days")
print(f"  Start : ${equity_curve['equity'].iloc[0]:,.2f}")
print(f"  End   : ${equity_curve['equity'].iloc[-1]:,.2f}")
print(f"  Change: ${equity_curve['equity'].iloc[-1] - STARTING_CAPITAL:+,.2f}")
print()
print("Sample (first 5 rows):")
print(equity_curve.head())

In [ ]:
fig = plot_equity_curve(equity_curve, trades, TICKER_A, TICKER_B, STARTING_CAPITAL)
fig.show()

**Reading the equity curve chart:**

The chart has two panels:

**Top panel — Portfolio equity:**
- The blue line shows portfolio value over time, starting at $20,000.
- The dashed horizontal line marks the starting capital — everything above it is profit, below is loss.
- **Green bands** mark periods when a long-spread trade was open; **red bands** mark short-spread trades. These show when capital was deployed.
- A healthy strategy shows the equity line generally climbing, with relatively shallow dips.

**Bottom panel — Drawdown:**
- Shows how far below its prior peak the portfolio sits at any point.
- Computed as `(equity / equity.cummax() - 1)` — a negative percentage.
- This is the "pain chart": it tells you the worst period you would have experienced if you had started trading at the worst possible time.
- A drawdown of -10% means the portfolio fell 10% from its highest point before recovering.

**What to notice on this pair:**
- Are the green/red bands followed by jumps in equity (exits) or drops (stops)?
- How long does the equity stay flat? Long flat periods mean positions are held open without crystallizing P&L.
- Is the drawdown panel roughly symmetric (losses recovered) or does it ratchet downward (losses compound)?

**The flat sections explained:** Between the trade bands, the equity curve is perfectly flat. When no position is open, no money is at risk and no P&L is generated. The curve only moves when a trade closes.

---

## Section 4: Performance Metrics

The metrics dictionary contains 10 statistics. This section works through the math and meaning of each one.

---

### Return Metrics

**Total Return:**
$$\text{total\_return} = \frac{\text{final equity} - \text{starting capital}}{\text{starting capital}}$$

Simple percentage gain or loss over the entire backtest period. Does not account for how long the period was.

**CAGR (Compound Annual Growth Rate):**
$$\text{CAGR} = \left(\frac{\text{final equity}}{\text{starting capital}}\right)^{252 / n_{\text{days}}} - 1$$

Annualizes the total return so you can compare strategies run over different time periods. Uses 252 trading days per year. A CAGR of 8% means the portfolio grew at the same rate as if it compounded at 8% per year.

---

### Risk-Adjusted Return

**Sharpe Ratio:**
$$\text{Sharpe} = \frac{\bar{r}_d}{\sigma_d} \times \sqrt{252}$$

Where $\bar{r}_d$ is the mean daily return and $\sigma_d$ is the standard deviation of daily returns.

The most widely used risk-adjusted performance metric. It answers: *"How much return did you earn per unit of volatility?"*

The $\sqrt{252}$ converts the daily ratio to an annualized figure (variance scales linearly with time, so standard deviation scales with the square root).

| Sharpe | Interpretation |
|---|---|
| > 2.0 | Excellent |
| 1.0 – 2.0 | Good — professional target for systematic strategies |
| 0.5 – 1.0 | Acceptable |
| 0.0 – 0.5 | Weak |
| < 0.0 | Negative — taking risk and losing money |

**Important note for sparse strategies:** Our equity curve is mostly flat (only changes on exit days), so the daily return series has many zeros. A mostly-flat series has near-zero variance, which can distort the Sharpe. For strategies with very few trades, the Sharpe is less informative than it is for high-frequency strategies. Use it as a rough guide, not a precise measure.

**Max Drawdown:**
$$\text{max\_drawdown} = \min_t \left(\frac{\text{equity}_t}{\max_{s \le t}(\text{equity}_s)} - 1\right)$$

At each point in time, divide the current equity by the highest equity seen so far (the high watermark). The minimum of this ratio minus 1 is the maximum drawdown. Always a negative number (or zero if no drawdown).

*"What was the worst peak-to-trough loss an investor would have experienced?"*

---

### Trade Statistics

**Win Rate:**
$$\text{win\_rate} = \frac{\text{trades with pnl} > 0}{\text{total trades}}$$

The fraction of trades that were profitable. Win rate alone is not sufficient — a strategy with a 30% win rate can still be very profitable if the average win is much larger than the average loss.

**Avg Win / Avg Loss:**
$$\text{avg\_win} = \text{mean}(\text{pnl} \mid \text{pnl} > 0)$$
$$\text{avg\_loss} = \text{mean}(\text{pnl} \mid \text{pnl} \le 0)$$

The ratio $|\text{avg\_win} / \text{avg\_loss}|$ is the **payoff ratio** — how many times larger the average win is versus the average loss. Together with win rate, these two numbers fully characterize the strategy's edge.

**Profit Factor:**
$$\text{profit\_factor} = \frac{\sum \text{pnl}_{\text{wins}}}{|\sum \text{pnl}_{\text{losses}}|}$$

Total gross profit divided by total gross loss. A profit factor > 1 means the strategy made more on winning trades than it lost on losing ones.

| Profit Factor | Interpretation |
|---|---|
| > 2.0 | Strong — for every $1 lost, $2 gained on average |
| 1.5 – 2.0 | Good — professional threshold |
| 1.0 – 1.5 | Marginal — small edge; transaction costs may erase it |
| < 1.0 | Unprofitable overall |

**Avg Hold Days:**
$$\text{avg\_hold} = \text{mean}(\text{exit\_date} - \text{entry\_date})$$

Average calendar days a position was held. Compare this to the half-life: if avg hold is much longer than the half-life, trades are spending a lot of time in drawdown before either reversing or getting stopped. If much shorter, the exit threshold may be too aggressive.

In [ ]:
def fmt_pct(v):  return f"{v:+.2%}" if v == v else "n/a"
def fmt_num(v):  return f"{v:+.2f}"  if v == v else "n/a"
def fmt_dollar(v): return f"${v:+,.2f}" if v == v else "n/a"
def fmt_pf(v):
    if v != v:  return "n/a"
    if v == float('inf'): return "inf (no losing trades)"
    return f"{v:.2f}"

print("=" * 52)
print(f"  BACKTEST RESULTS  {TICKER_A}/{TICKER_B}")
print("=" * 52)
print()
print("-- RETURN METRICS --------------------------------")
print(f"  Total return   : {fmt_pct(metrics['total_return'])}")
print(f"  CAGR           : {fmt_pct(metrics['cagr'])}")
print()
print("-- RISK METRICS ----------------------------------")
print(f"  Sharpe ratio   : {fmt_num(metrics['sharpe'])}")
print(f"  Max drawdown   : {fmt_pct(metrics['max_drawdown'])}")
print()
print("-- TRADE STATISTICS ------------------------------")
print(f"  Trades         : {metrics['n_trades']}")
print(f"  Win rate       : {fmt_pct(metrics['win_rate'])}")
print(f"  Avg win        : {fmt_dollar(metrics['avg_win'])}")
print(f"  Avg loss       : {fmt_dollar(metrics['avg_loss'])}")
print(f"  Profit factor  : {fmt_pf(metrics['profit_factor'])}")
print(f"  Avg hold days  : {fmt_num(metrics['avg_hold_days'])}")
print()

# Sanity check: sum of trade P&L should match equity change
if not trades.empty:
    pnl_sum = trades['pnl'].sum()
    equity_delta = equity_curve['equity'].iloc[-1] - STARTING_CAPITAL
    print(f"  Cross-check: sum of trade P&L   = ${pnl_sum:+,.2f}")
    print(f"               equity curve delta = ${equity_delta:+,.2f}")
    print(f"               match              = {abs(pnl_sum - equity_delta) < 0.01}")

**Interpreting these results:**

Given that KO/PEP is not stationary over this window, we expect poor results — and the backtest confirms it. This is an important validation: the strategy correctly produces negative returns when its core assumption (stationarity) is violated.

A few things to notice:

**The Sharpe and drawdown relationship:**
A negative Sharpe alongside a large max drawdown means we are not just losing money — we are losing it with significant volatility. This is worse than simply holding cash.

**Trade count and statistical significance:**
With fewer than 20 trades, the win rate and profit factor statistics have high uncertainty. You could get a 60% win rate by luck with 5 trades. Statistical significance for these metrics generally requires 30+ trades. Treat small-sample results with healthy skepticism.

**The sum check:**
The cross-check above confirms that the equity curve is internally consistent — the sum of all trade P&L should equal the total change in equity. This sanity check is useful when debugging a backtest.

**Comparing to the half-life:**
The half-life is typically very long for this pair (~110 days), but the default rolling window is only 60 days. When the window is shorter than the half-life, the z-score over-fires — it flags "unusual" deviations that are actually just normal parts of a slow cycle. This mismatch between window and half-life is another reason this pair performs poorly even in regimes where it might have some signal.

In [ ]:
params_dict = {
    'period': p['period'],
    'rolling_window': p['rolling_window'],
    'z_entry': p['z_entry'],
    'z_exit': p['z_exit'],
    'z_stop': p['z_stop'],
}

plot_backtest_metrics(metrics, trades, TICKER_A, TICKER_B, params_dict).show()

In [ ]:
plot_backtest_interpretation(metrics, trades, TICKER_A, TICKER_B, params_dict, hl).show()

---

## Section 5: Trade P&L Distribution

The equity curve shows the cumulative story. The trade P&L chart breaks it down trade by trade — which trades made money, which lost, and by how much.

### What the distribution reveals

Beyond the summary statistics (win rate, avg win/loss), the full distribution of P&L per trade tells you:

**Are wins and losses roughly symmetric?**
In a mean-reverting strategy with good stop-losses, winning trades (spread reverted) and losing trades (stop hit) should each cover a consistent range. If losses are dramatically larger than wins, the stop level may be too wide.

**Are there outliers?**
A single large winner or loser can dominate the statistics. A strategy with 7 small wins and 1 catastrophic loss may look profitable on win rate but be negative on total return. Look at the full distribution, not just the averages.

**What fraction of trades were stopped out vs. exited normally?**
A high stop rate (more than 40% of trades closing via stop) suggests either the stop is too tight for the pair's typical volatility, or the pair is trending rather than mean-reverting. A good mean-reversion strategy should have most trades closing via exit (spread reverted) rather than stop (spread kept moving away).

In [ ]:
plot_trade_pnl(trades, TICKER_A, TICKER_B).show()

In [ ]:
# Breakdown by exit type
if not trades.empty:
    print("Trade breakdown by exit type:")
    for exit_type, group in trades.groupby('exit_type'):
        print(f"  {exit_type:<8}: {len(group)} trades  |  "
              f"total P&L {group['pnl'].sum():+,.2f}  |  "
              f"avg {group['pnl'].mean():+,.2f}")
    print()
    stop_rate = (trades['exit_type'] == 'stop').mean()
    print(f"  Stop rate: {stop_rate:.0%}")
    if stop_rate > 0.4:
        print()
        print("  High stop rate. This often indicates the spread is trending,")
        print("  the stop threshold is too tight, or the pair is not truly mean-reverting.")
else:
    print("No trades.")

---

## Section 6: Limitations of this Backtest

Understanding what a backtest *cannot* tell you is just as important as understanding what it can. This section catalogs the known simplifications in the current implementation and explains what would need to change for a production-grade system.

---

### 1. Close-to-Close Fills (Signal Lag)

**What we do:** Enter at the close on the signal bar. The z-score crosses the threshold on day T, and we assume we traded at the close price on day T.

**Why this is optimistic:** In practice, you observe the close price *after* the market closes. You cannot trade at the price you are using to generate the signal. The realistic implementation is **signal on close T, fill at open T+1** (next-bar execution). Close prices are often revised in after-hours trading; opening prices may gap away from the prior close.

**What to change:** In `build_trade_log`, shift the fill prices by one bar forward. Use `df_ohlcv['open_a'].shift(-1)` as the entry price instead of `close_a`. This typically reduces returns slightly (slippage from close-to-open moves) but produces a more honest simulation.

---

### 2. Static Hedge Ratio (Look-Ahead Bias)

**What we do:** Estimate the hedge ratio beta using the full 2-year dataset, then apply it to the entire historical signal series.

**Why this is problematic:** The hedge ratio estimated on all data uses *future* information when generating signals for early dates. On day 1 of the backtest, we are using a beta that incorporates data from 2 years ahead — data that could not have been known at the time.

**What to change:** Replace `compute_hedge_ratio` with a `compute_rolling_hedge_ratio` function that, at each point in time, estimates beta using only the trailing N days (e.g., 252 days). The spread and z-score then use this rolling beta. This is the standard approach in production pairs trading systems.

---

### 3. No Transaction Costs or Slippage

**What we do:** Assume zero cost to enter or exit any position.

**The real costs:** Most brokers charge $0 commission for stock trades, but the bid-ask spread is a real cost. For liquid large-caps like KO/PEP, the bid-ask spread is typically 1-2 cents — about 0.02% per side, or roughly $4 on a $10,000 leg. For a round-trip (entry + exit), that is $8-16 per trade pair. Over a strategy with 15 trades, that is $120-240 in costs — meaningful relative to total P&L.

**What to change:** Add a `slippage_bps` parameter to `run_pairs_backtest`. Reduce the effective sell price and increase the effective buy price by `slippage_bps * 0.01%`. For KO/PEP, 5-10 bps round-trip is a realistic estimate.

---

### 4. In-Sample Testing (Overfitting Risk)

**What we do:** Generate signals and run the backtest using the same date range that was used to estimate the hedge ratio and confirm stationarity.

**Why this is a problem:** If you tune z_entry, z_exit, z_stop, and the rolling window until the backtest looks good, you have learned the specific noise in your historical data — not a generalizable edge. The backtest will look much better than the strategy will actually perform.

**What to change:** Walk-forward validation splits the data into sequential windows:
1. **Training window:** Estimate beta, confirm ADF, choose parameters.
2. **Test window:** Run the backtest on this unseen period with parameters locked from training.
3. Roll both windows forward and repeat.

Average performance across test windows is the honest estimate of out-of-sample performance.

---

### 5. Unrealized P&L Not Tracked

The equity curve only moves on exit days. This means the max drawdown may be understated — if a position was running at a large unrealized loss before being stopped out, the equity curve does not reflect that until the stop fires. For a more accurate picture, you would mark positions to market daily (compute unrealized P&L at each day's close while the position is open).

---

### 6. No Portfolio-Level Risk Management

**What we do:** Size each trade at a fixed $10k/leg regardless of market conditions.

**Better approaches:**
- **Volatility-scaled sizing:** Reduce position size when the spread's recent volatility is unusually high.
- **Kelly criterion:** Size based on estimated edge and variance.
- **Correlation-aware sizing:** If trading multiple pairs simultaneously, account for the correlation between them — two financial-sector pairs are not independent bets.

---

## Summary

### The full pipeline

```
Signal DataFrame (from Notebook 01)
         |
         v
build_trade_log()
  - pairs each entry with its exit/stop
  - sizes positions: dollar-neutral, hedge-ratio adjusted
  - computes per-trade P&L
         |
         v
run_backtest()
  - groups P&L by exit date
  - builds daily equity curve via cumsum
  - fills flat periods for un-traded days
         |
         v
compute_metrics()
  - total return, CAGR, Sharpe, max drawdown
  - win rate, avg win/loss, profit factor, avg hold days
         |
         v
Visualizations
  - plot_equity_curve()            portfolio value + drawdown
  - plot_trade_pnl()               per-trade P&L bar chart
  - plot_backtest_metrics()        summary stats table
  - plot_backtest_interpretation() plain-English explanation
```

### Key formulas

| Metric | Formula |
|---|---|
| Trade P&L | $\text{shares}_A(\text{exit}_A - \text{entry}_A) + \text{shares}_B(\text{exit}_B - \text{entry}_B)$ |
| Total return | $(\text{final} - \text{start}) / \text{start}$ |
| CAGR | $(\text{final}/\text{start})^{252/n} - 1$ |
| Sharpe | $\bar{r}_d / \sigma_d \times \sqrt{252}$ |
| Max drawdown | $\min_t(\text{equity}_t / \text{max equity}_{\le t} - 1)$ |
| Profit factor | $\sum \text{wins} / |\sum \text{losses}|$ |

### What to add as the system grows

This notebook is intentionally modular. Add new sections as new capabilities are built:

- **Section 7** — Rolling hedge ratio comparison (static vs. rolling beta side-by-side)
- **Section 8** — Walk-forward validation (out-of-sample performance)
- **Section 9** — Transaction cost sensitivity (how does Sharpe change at 0 / 5 / 10 bps?)
- **Section 10** — Position sizing experiments (fixed vs. z-score-scaled vs. volatility-scaled)
- **Section 11** — Multi-pair portfolio (two valid pairs simultaneously; measure correlation of returns)

### The most important next step

Find pairs that actually cointegrate. The backtest is working correctly — it correctly shows negative results for a non-stationary pair. The framework is ready; it just needs better inputs.

Strong candidates to test:
- `GS/MS` — Goldman Sachs vs Morgan Stanley
- `DAL/UAL` — Delta vs United Airlines
- `CVS/WBA` — pharmacy retail duopoly
- Any of the current pairs with `--period 6mo` or `--period 1y` (a shorter window may capture a valid cointegration regime)